## Process the ProteinGLUE CSVs

In [1]:
import pickle
import pandas as pd
import re
import os

In [2]:
dfs = [] # will store the dataframes for each CSV
fns = [] # will store the filenames of each CSV
for fn in os.listdir("../datasets/ProteinGLUE"):
    if 'csv' in fn: # only read CSV files
        df = pd.read_csv("../datasets/ProteinGLUE" + "/" + fn)

        # remove the brackets, quotes, and newlines from the labels
        for col in df.columns:
            df[col] = df[col].apply(lambda x: re.sub("[\[\]\'b\n]", '', x))
        dfs.append(df)
        fns.append(fn)

names = [fn[:-4] for fn in fns] # remove the .csv from the filename to get the dataset name

datasets = {} # will store the datasets
for name, df in zip(names, dfs):
    datasets[name] = {} # initialize the dataset

    datasets[name]['sequences'] = [] # will store the sequences
    label_columns = [item for item in df.columns.tolist() if item != 'sequence'] # get the label columns, excluding the sequence column
    error_indices = [] # will store the indices of the labels with errors so we don't append the corresponding sequence to the dataset

    for label_column in label_columns: # iterate through the label columns
        raw_labels = df[label_column].tolist() # get the raw labels

        num_errors = 0 # will store the number of labels with errors
        labels = [] # will store the labels
        for i, raw_label in enumerate(raw_labels):
            if "..." in raw_label.split(): # some labels have "..." in them for some reason
                num_errors += 1
                error_indices.append(i) # add the index of the label with an error
                continue # skip this label

            labels.append([float(item) for item in raw_label.split()]) # convert the label to a list of floats and append it to the labels list
        
        datasets[name][label_column] = labels # add the labels to the dataset
        print("Dataset: {}, Label: {}, Num Errors: {}".format(name, label_column, num_errors)) # print the number of errors for this label
    
    # add the sequences to the dataset
    raw_sequences = df['sequence'].to_list() # get the raw sequences
    for i in range(0, len(raw_sequences)):
        if i not in error_indices: # if the label at this index doesn't have an error
            datasets[name]['sequences'].append(raw_sequences[i])

    # verify the dataset
    for label in datasets[name].keys():
        # check that the number of labels is the same as the number of sequences
        if len(datasets[name][label]) != len(datasets[name]['sequences']):
            print(f"ERROR {name} {label} {len(datasets[name][label])} {len(datasets[name]['sequences'])}")
            continue

        # check that the length of each label is the same as the length of the corresponding sequence
        for i in range(0, len(datasets[name][label])):
            if len(datasets[name][label][i]) != len(datasets[name]["sequences"][i]):
                print(f"ERROR {name} {label} {i} {len(datasets[name][label][i])} {len(datasets[name]['sequences'][i])}")
    
    print() # print a newline

Dataset: Epitope_anti_validation_1, Label: interface, Num Errors: 0

Dataset: HPrank_validation, Label: hp_class, Num Errors: 2
Dataset: HPrank_validation, Label: hydrophobic_patch, Num Errors: 2
Dataset: HPrank_validation, Label: hp_rank, Num Errors: 2

Dataset: ppi_hetro_homo_validation, Label: interface, Num Errors: 0

Dataset: ppi_hetro_homo_test, Label: interface, Num Errors: 1

Dataset: Epitope_anti_test_1, Label: interface, Num Errors: 2

Dataset: asabu_training, Label: buried, Num Errors: 0
Dataset: asabu_training, Label: solvent_accessibility, Num Errors: 0

Dataset: ss_test, Label: ss3, Num Errors: 0
Dataset: ss_test, Label: ss8, Num Errors: 0

Dataset: ppi_hetro_homo_training, Label: interface, Num Errors: 1

Dataset: ss_validation, Label: ss3, Num Errors: 0
Dataset: ss_validation, Label: ss8, Num Errors: 0

Dataset: ss_training, Label: ss3, Num Errors: 0
Dataset: ss_training, Label: ss8, Num Errors: 0

Dataset: HPrank_test, Label: hp_rank, Num Errors: 5
Dataset: HPrank_test

In [3]:
# save dataset
with open("../datasets/ProteinGLUE_processed.pkl", "wb") as f:
    pickle.dump(datasets, f)

# Check reconstruction accuracy of tokenized labels

In [4]:
from tqdm import tqdm
import sentencepiece as spm
import numpy as np

sp = spm.SentencePieceProcessor()
sp.Load("../training/m.model")

def get_accuracy(sequences, labels):
    load_errors = []

    def format_sequence_and_label(sequence, label):
        tokenized_sequence = sp.EncodeAsIds(sequence)
        if tokenized_sequence[0] == 38:
            tokenized_sequence = tokenized_sequence[1:]
        
        token_lens = [len(sp.DecodeIds(token)) for token in tokenized_sequence]
        
        label_modes = []
        ptr = 0
        for length in token_lens:
            label_modes.append(np.bincount(label[ptr:ptr+length]).argmax())
            ptr += length
        sequence_length = len(sequence)
        return tokenized_sequence, label_modes, token_lens, sequence_length

    def regenerate_label(label_modes, token_lens):
        label = []
        for mode, length in zip(label_modes, token_lens):
            label.extend([mode] * length)
        return label

    def get_accuracy(prediction, label):
        prediction = np.array(prediction)
        label = np.array(label)

        if len(prediction) != len(label):
            raise ValueError("Prediction and label must have the same length.")
        return np.sum(prediction == label) / len(prediction)


    regenerated_accs = []
    regen_errors = []
    seq_lens = []
    for i in tqdm(range(0, len(sequences))):
        try:
            if len(sequences[i]) != len(labels[i]):
                print(len(sequences[i]), len(labels[i]))
            tokenized_sequence, label_modes, token_lens, sequence_length = format_sequence_and_label(sequences[i], labels[i])
                
            regenerated_accs.append(get_accuracy(regenerate_label(label_modes, token_lens), labels[i])*sequence_length)
            seq_lens.append(sequence_length)

        except:
            regen_errors.append(i)
    
    print(np.sum(regenerated_accs)/np.sum(seq_lens))
    print(f"Num load errors: {len(load_errors)}, num regen errors: {len(regen_errors)}")

In [5]:
for name in datasets.keys():
    print(name)
    for label in datasets[name].keys():
        if label != 'sequences':
            print(label)
            get_accuracy(datasets[name]['sequences'], datasets[name][label])
    print()

asabu_test
buried


  0%|          | 0/1102 [00:00<?, ?it/s]

100%|██████████| 1102/1102 [00:00<00:00, 2027.28it/s]


0.8441871289635068
Num load errors: 0, num regen errors: 0
solvent_accessibility


100%|██████████| 1102/1102 [00:00<00:00, 2033.02it/s]


...

0.8613774073771829
Num load errors: 0, num regen errors: 0

ss_validation
ss3


100%|██████████| 1102/1102 [00:00<00:00, 1934.70it/s]


0.9203665325693852
Num load errors: 0, num regen errors: 0
ss8


100%|██████████| 1102/1102 [00:00<00:00, 1911.92it/s]

0.8598642772969239
Num load errors: 0, num regen errors: 0

